# Drifting Models — native PyTorch inference

This notebook converts the official JAX checkpoint once when needed, then runs only the native PyTorch generator. The JAX and PyTorch packages remain explicit; no default backend is selected.

In [ ]:
import os, subprocess, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/RahulBhalley/drifting.git'
REPO_REF = 'main'
ROOT = Path('/content/drifting') if IN_COLAB else Path.cwd()
if IN_COLAB and not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, str(ROOT)], check=True)
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{ROOT}[parity]'], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('repository:', ROOT)

In [ ]:
MODEL_NAME = 'pixel_B_sota'
HF_CACHE = ROOT / 'artifacts' / 'hf-cache'
ARTIFACT = Path(os.environ.get('DRIFTING_NOTEBOOK_ARTIFACT', ROOT / 'artifacts' / 'torch' / MODEL_NAME)).resolve()
if not (ARTIFACT / 'manifest.json').is_file():
    subprocess.run([sys.executable, 'tools/convert_checkpoint.py', '--kind', 'generator', '--source', f'hf://{MODEL_NAME}', '--output', str(ARTIFACT), '--hf-cache', str(HF_CACHE)], check=True)
print('PyTorch artifact:', ARTIFACT)

In [ ]:
from drifting_torch.inference import InferenceRequest, generate

CLASS_IDS = (95, 22, 88, 108, 386, 296, 483, 698)
result = generate(InferenceRequest(source=ARTIFACT, class_ids=CLASS_IDS, cfg_scale=1.0, seed=0, device='auto', precision='fp32', output_dir=ROOT / 'outputs' / 'torch-notebook'))
print(result.metadata)

In [ ]:
from PIL import Image, ImageDraw

thumbs = []
for path, label in zip(result.image_paths, CLASS_IDS):
    image = Image.open(path).resize((256, 256))
    canvas = Image.new('RGB', (256, 280), 'white')
    canvas.paste(image, (0, 24))
    ImageDraw.Draw(canvas).text((8, 5), f'class {label}', fill='black')
    thumbs.append(canvas)
grid = Image.new('RGB', (1024, 560), 'white')
for index, image in enumerate(thumbs):
    grid.paste(image, ((index % 4) * 256, (index // 4) * 280))
if IN_COLAB:
    from IPython.display import display
    display(grid)
else:
    print('generated:', *result.image_paths, sep='\n')
assert result.metadata['sample_finite'] and result.metadata['sample_shape'] == [8, 3, 256, 256]